In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HTTP_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["HTTPS_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["http_proxy"]= "http://proxy.utwente.nl:3128"
os.environ["https_proxy"]= "http://proxy.utwente.nl:3128"
import os
import json
import random
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics.pairwise import cosine_similarity

Unsupervised

In [2]:

TEST_DATA_PATH = "/home/simonettos/thijs/data_augmentatio_stefano/siamese_unsup_sup/test_clean_tram2.json"
TTP_DESC_PATH   = "/home/simonettos/thijs/data_augmentatio_stefano/mitre/ttp_descriptions2.json"
BASE_MODEL = "ehsanaghaei/SecureBERT"

BATCH_SIZE = 32
PATIENCE   = 5
SEED       = 42

MAX_LEN_SENT = 192
MAX_LEN_TTP  = 256

LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_EPOCHS = 50
TEMPERATURE = 0.05

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# REPRODUCIBILITY
# -------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# -------------------------
# LOAD DATA
# -------------------------

with open(TTP_DESC_PATH, "r", encoding="utf-8") as f:
    ttp_descriptions = json.load(f)


In [3]:
with open(TEST_DATA_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)   # list of {"sentence":..., "labels":[...]}

# -------------------------
# DATASET + COLLATE
# -------------------------
class PairDataset(Dataset):
    def __init__(self, pairs: List[Tuple[str, str]]):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx: int):
        return self.pairs[idx]  # (sent, ttp_text)

@dataclass
class DualCollator:
    tokenizer: object
    max_len_sent: int
    max_len_ttp: int

    def __call__(self, batch: List[Tuple[str, str]]) -> Dict[str, Dict[str, torch.Tensor]]:
        sents = [b[0] for b in batch]
        ttps  = [b[1] for b in batch]

        tok_sent = self.tokenizer(
            sents,
            padding=True,
            truncation=True,
            max_length=self.max_len_sent,
            return_tensors="pt",
        )
        tok_ttp = self.tokenizer(
            ttps,
            padding=True,
            truncation=True,
            max_length=self.max_len_ttp,
            return_tensors="pt",
        )
        return {"sent": tok_sent, "ttp": tok_ttp}

# -------------------------
# MODEL: SecureBERT bi-encoder
# -------------------------
class SecureBertEmbedder(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)

    @staticmethod
    def mean_pool(last_hidden: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
        summed = (last_hidden * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-9)
        return summed / denom

    def encode_batch(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        emb = self.mean_pool(out.last_hidden_state, attention_mask)
        emb = F.normalize(emb, p=2, dim=1)
        return emb

    @torch.no_grad()
    def encode_texts(self, tokenizer, texts: List[str], batch_size: int, max_length: int) -> np.ndarray:
        self.eval()
        all_embs = []
        for i in range(0, len(texts), batch_size):
            chunk = texts[i:i+batch_size]
            tok = tokenizer(
                chunk,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            tok = {k: v.to(DEVICE) for k, v in tok.items()}
            embs = self.encode_batch(tok["input_ids"], tok["attention_mask"])
            all_embs.append(embs.detach().cpu().numpy())
        return np.vstack(all_embs)

# -------------------------
# LOSS: In-batch negatives
# -------------------------
class InBatchNegativesLoss(nn.Module):
    def __init__(self, temperature: float = 0.05):
        super().__init__()
        self.temperature = temperature

    def forward(self, emb_a: torch.Tensor, emb_b: torch.Tensor) -> torch.Tensor:
        logits = (emb_a @ emb_b.t()) / self.temperature
        labels = torch.arange(logits.size(0), device=logits.device)
        return F.cross_entropy(logits, labels)


# -------------------------
# INIT MODEL / OPT / SCHED
# -------------------------
model = SecureBertEmbedder(BASE_MODEL).to(DEVICE)
loss_fn = InBatchNegativesLoss(temperature=TEMPERATURE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
# -------------------------
# PREP VAL TTP TEXTS
# -------------------------
ttp_ids = [t for t in ttp_descriptions.keys()]
ttp_texts = [f"{t}: {ttp_descriptions[t]}" for t in ttp_ids]
# -------------------------
# VALIDATION FUNCTION
# -------------------------
def get_val_sentence(item):
    if isinstance(item, dict):
        return (item.get("sentence") or "").strip()
    if isinstance(item, (list, tuple)):
        return (item[0] or "").strip()
    raise TypeError(f"Unsupported val item type: {type(item)}")

def get_val_labels(item):
    if isinstance(item, dict):
        return item.get("labels", []) or []
    if isinstance(item, (list, tuple)):
        return item[1] if len(item) > 1 else []
    raise TypeError(f"Unsupported val item type: {type(item)}")

def validate(model: SecureBertEmbedder, k_list=(1, 5, 10)) -> Dict[str, float]:
    model.eval()

    ttp_embs = model.encode_texts(tokenizer, ttp_texts, batch_size=64, max_length=MAX_LEN_TTP)

    hits_at_k = {k: 0 for k in k_list}
    mean_recall_at_k = {k: 0.0 for k in k_list}
    mrr = 0.0
    used = 0

    for item in val_data:
        sent = get_val_sentence(item)
        raw_lbls = get_val_labels(item)

        gold = {t.strip() for t in raw_lbls if isinstance(t, str) and t.strip()}
        gold = {t for t in gold if t in ttp_descriptions.keys()}
        if not sent or not gold:
            continue

        used += 1

        sent_emb = model.encode_texts(tokenizer, [sent], batch_size=1, max_length=MAX_LEN_SENT)[0]
        sims = cosine_similarity(sent_emb.reshape(1, -1), ttp_embs)[0]
        ranking = np.argsort(-sims)

        rr = 0.0
        for rank_pos, idx in enumerate(ranking, start=1):
            if ttp_ids[idx] in gold:
                rr = 1.0 / rank_pos
                break
        mrr += rr

        for k in k_list:
            topk = [ttp_ids[i] for i in ranking[:k]]
            hits_at_k[k] += int(any(t in gold for t in topk))
            mean_recall_at_k[k] += len(set(topk) & gold) / max(1, len(gold))

    n = max(1, used)
    metrics = {f"hit@{k}": hits_at_k[k] / n for k in k_list}
    metrics.update({f"mean_recall@{k}": mean_recall_at_k[k] / n for k in k_list})
    metrics["mrr"] = mrr / n
    metrics["val_used"] = used
    return metrics



# -------------------------
# SAVE HELPERS
# -------------------------
def save_model(model: SecureBertEmbedder, tokenizer, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    model.backbone.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
model = SecureBertEmbedder(BASE_MODEL).to(DEVICE)

metrics = validate(model, k_list=(1, 5, 10))
print("UNSUPERVISED baseline (pretrained SecureBERT bi-encoder)")
print(", ".join([f"{k}: {v:.4f}" for k, v in metrics.items()]))

Some weights of RobertaModel were not initialized from the model checkpoint at ehsanaghaei/SecureBERT and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at ehsanaghaei/SecureBERT and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


UNSUPERVISED baseline (pretrained SecureBERT bi-encoder)
hit@1: 0.0534, hit@5: 0.0957, hit@10: 0.1218, mean_recall@1: 0.0518, mean_recall@5: 0.0913, mean_recall@10: 0.1162, mrr: 0.0803, val_used: 2414.0000


In [5]:
# -------------------------
# PREP VAL TTP TEXTS (allowed labels = all described labels)
# -------------------------
allowed_set = set(ttp_descriptions.keys())      # explicit "allowed labels"
ttp_ids = sorted(allowed_set)                  # deterministic order
ttp_texts = [f"{t}: {ttp_descriptions[t]}" for t in ttp_ids]

print(f"[Eval] Allowed labels (described): {len(allowed_set)}")
print("[Eval] First 30 allowed labels:", ttp_ids[:30])

# If you want to print ALL (can be huge)
# for t in ttp_ids:
#     print(t)


[Eval] Allowed labels (described): 836
[Eval] First 30 allowed labels: ['T1001', 'T1001.001', 'T1001.002', 'T1001.003', 'T1002', 'T1003', 'T1003.001', 'T1003.002', 'T1003.003', 'T1003.004', 'T1003.005', 'T1003.006', 'T1003.007', 'T1003.008', 'T1004', 'T1005', 'T1006', 'T1007', 'T1008', 'T1009', 'T1010', 'T1011', 'T1011.001', 'T1012', 'T1013', 'T1014', 'T1015', 'T1016', 'T1016.001', 'T1016.002']
